# Failure Forensics

Structured forensic investigation of model failure behavior under split-level spectral variation.

This notebook is the third stage of the research workflow. It focuses on where the model fails, whether those failures are structured, which classes absorb misclassifications, and how those pathways appear in morphology and latent space.

## Notebook Identity

The intent here is evidence-first analysis. The notebook investigates confusion structure, attractor behavior, morphology alignment, and embedding overlap without jumping to abstract interpretation or mitigation proposals.

Primary inputs: evaluation JSON, prediction arrays, embeddings, reference split spectra, test split spectra, labels, and confusion matrices.

Before running the analysis, set `EXPERIMENT_DIR` to the experiment root that contains the exported evaluation artifacts.

In [1]:
from pathlib import Path
import json
from typing import Any, Dict, List, Optional, Sequence, Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, confusion_matrix, matthews_corrcoef, precision_recall_fscore_support, roc_auc_score
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.neighbors import NearestNeighbors

try:
    import umap
except ImportError:
    umap = None

sns.set_theme(style='whitegrid', context='talk')
mpl.rcParams.update({
    'figure.dpi': 140,
    'savefig.dpi': 220,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'axes.titlesize': 13,
    'legend.frameon': False,
    'font.size': 10,
})
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 140)

In [2]:
def find_project_root(start: Optional[Path] = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    return current


def load_json(path: Path) -> Optional[Dict[str, Any]]:
    if path.exists():
        return json.loads(path.read_text(encoding='utf-8'))
    return None


def load_mapping(path: Path) -> Optional[Dict[str, Any]]:
    if not path.exists():
        return None
    if path.suffix.lower() == '.json':
        return load_json(path)
    if path.suffix.lower() in {'.yaml', '.yml'}:
        try:
            import yaml
        except ImportError:
            return None
        with path.open('r', encoding='utf-8') as handle:
            return yaml.safe_load(handle)
    return None


def discover_eval_results(exp_dir: Path) -> List[Path]:
    candidates: List[Path] = []
    for root in [exp_dir, exp_dir / 'analysis', exp_dir.parent / 'analysis']:
        if root.exists():
            candidates.extend(root.rglob('*_eval_results.json'))
    unique: List[Path] = []
    seen = set()
    for path in sorted(candidates):
        resolved = path.resolve()
        if resolved not in seen:
            unique.append(path)
            seen.add(resolved)
    return unique


def discover_bundle_splits(directory: Path, suffix: str) -> List[str]:
    if not directory.exists():
        return []
    return sorted({path.name[:-len(suffix)] for path in directory.glob(f'*{suffix}')})


def load_prediction_bundle(pred_dir: Path, split: str) -> Optional[Dict[str, np.ndarray]]:
    bundle: Dict[str, np.ndarray] = {}
    for key in ['logits', 'probabilities', 'predictions', 'targets']:
        path = pred_dir / f'{split}_{key}.npy'
        if path.exists():
            bundle[key] = np.load(path)
    return bundle or None


def load_embedding_bundle(emb_dir: Path, split: str) -> Optional[Dict[str, np.ndarray]]:
    bundle: Dict[str, np.ndarray] = {}
    for key in ['features', 'targets']:
        path = emb_dir / f'{split}_{key}.npy'
        if path.exists():
            bundle[key] = np.load(path)
    return bundle or None


def load_raw_split(data_root: Path, split: str) -> Optional[Tuple[np.ndarray, np.ndarray]]:
    x_path = data_root / f'X_{split}.npy'
    y_path = data_root / f'y_{split}.npy'
    if x_path.exists() and y_path.exists():
        return np.load(x_path), np.load(y_path)
    return None


def extract_class_names(payload: Optional[Dict[str, Any]], targets: Sequence[Any]) -> List[str]:
    if isinstance(payload, dict):
        for key in ['class_names', 'classes', 'labels', 'target_names']:
            value = payload.get(key)
            if isinstance(value, list) and value:
                return [str(item) for item in value]
    try:
        uniques = sorted({int(item) for item in np.asarray(targets).ravel()})
        return [f'class_{item}' for item in uniques]
    except Exception:
        uniques = sorted({str(item) for item in np.asarray(targets).ravel()})
        return list(uniques)


def flatten_numeric_metrics(node: Any, prefix: str = '') -> Dict[str, float]:
    metrics: Dict[str, float] = {}
    if isinstance(node, dict):
        for key, value in node.items():
            full_key = f'{prefix}.{key}' if prefix else key
            if isinstance(value, dict):
                metrics.update(flatten_numeric_metrics(value, full_key))
            elif isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool):
                metrics[full_key] = float(value)
    return metrics


def compute_split_metrics(y_true: np.ndarray, y_pred: np.ndarray, probabilities: Optional[np.ndarray] = None) -> Dict[str, float]:
    metrics = {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'mcc': float(matthews_corrcoef(y_true, y_pred)),
    }
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    metrics['precision_macro'] = float(precision)
    metrics['recall_macro'] = float(recall)
    metrics['f1_macro'] = float(f1)
    if probabilities is not None and probabilities.ndim == 2 and probabilities.shape[1] > 1:
        try:
            metrics['roc_auc_ovr'] = float(roc_auc_score(y_true, probabilities, multi_class='ovr', average='macro'))
        except Exception:
            pass
    return metrics


def compute_class_failure_table(y_true: np.ndarray, y_pred: np.ndarray, class_names: Sequence[str]) -> Tuple[pd.DataFrame, np.ndarray, List[int], List[str]]:
    labels = sorted({int(value) for value in np.concatenate([y_true, y_pred])})
    display_names = [class_names[label] if label < len(class_names) else str(label) for label in labels]
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    rows: List[Dict[str, Any]] = []
    for index, label in enumerate(labels):
        row = cm[index].astype(float)
        total = int(row.sum())
        correct = int(row[index])
        errors = total - correct
        off = row.copy()
        off[index] = 0.0
        if errors > 0:
            dominant_index = int(off.argmax())
            dominant_label = labels[dominant_index]
            dominant_share = float(off[dominant_index] / errors)
            ranked_off = np.sort(off)[::-1]
            second_share = float(ranked_off[1] / errors) if len(ranked_off) > 1 and ranked_off[1] > 0 else 0.0
        else:
            dominant_label = label
            dominant_share = 0.0
            second_share = 0.0
        rows.append({
            'class_index': label,
            'class_name': display_names[index],
            'support': int(support[index]),
            'precision': float(precision[index]),
            'recall': float(recall[index]),
            'f1': float(f1[index]),
            'correct': correct,
            'errors': errors,
            'error_rate': float(errors / total) if total else 0.0,
            'dominant_confusion_index': int(dominant_label),
            'dominant_confusion_target': class_names[dominant_label] if dominant_label < len(class_names) else str(dominant_label),
            'dominant_confusion_share': dominant_share,
            'second_confusion_share': second_share,
            'confusion_concentration': dominant_share,
            'dominance_ratio': float(dominant_share / second_share) if second_share > 0 else np.inf,
        })
    table = pd.DataFrame(rows).sort_values(['f1', 'support'], ascending=[True, False]).reset_index(drop=True)
    return table, cm, labels, display_names


def confusion_pathways(cm: np.ndarray, labels: List[int], class_names: Sequence[str]) -> pd.DataFrame:
    total = float(cm.sum()) if cm.sum() else 1.0
    rows: List[Dict[str, Any]] = []
    for source_index, source_label in enumerate(labels):
        row_total = float(cm[source_index].sum()) if cm[source_index].sum() else 1.0
        for target_index, target_label in enumerate(labels):
            if source_index == target_index or cm[source_index, target_index] == 0:
                continue
            rows.append({
                'source_index': source_label,
                'source_name': class_names[source_label] if source_label < len(class_names) else str(source_label),
                'target_index': target_label,
                'target_name': class_names[target_label] if target_label < len(class_names) else str(target_label),
                'count': int(cm[source_index, target_index]),
                'row_share': float(cm[source_index, target_index] / row_total),
                'absolute_share': float(cm[source_index, target_index] / total),
            })
    return pd.DataFrame(rows).sort_values(['row_share', 'count'], ascending=[False, False]).reset_index(drop=True)


def label_name(value: Any, class_names: Sequence[str]) -> str:
    try:
        index = int(value)
        if 0 <= index < len(class_names):
            return class_names[index]
    except Exception:
        pass
    return str(value)


def build_embedding_frame(features: np.ndarray, targets: np.ndarray, split: str, class_names: Sequence[str], predictions: Optional[np.ndarray] = None) -> pd.DataFrame:
    frame = pd.DataFrame(features, columns=[f'dim_{index + 1}' for index in range(features.shape[1])])
    frame['split'] = split
    frame['true_index'] = targets.astype(int)
    frame['true_label'] = [label_name(value, class_names) for value in targets]
    if predictions is not None:
        frame['pred_index'] = predictions.astype(int)
        frame['pred_label'] = [label_name(value, class_names) for value in predictions]
        frame['correct'] = frame['true_index'] == frame['pred_index']
    else:
        frame['pred_index'] = np.nan
        frame['pred_label'] = None
        frame['correct'] = np.nan
    return frame


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / 'data' / 'raw'
EXPERIMENT_DIR = PROJECT_ROOT / 'path_to_your_stage1_tcn_supcon_experiment'
REFERENCE_SPLIT = 'reference'
TEST_SPLIT = 'test'

## 1. Load Evaluation Artifacts

Load the exported evaluation JSON, prediction bundles, embeddings, and raw reference/test spectra. The first pass is intentionally mechanical: identify what is available, summarize the experiment, and only then move into failure analysis.

In [3]:
EVAL_RESULTS = discover_eval_results(EXPERIMENT_DIR)
EVAL_PATH = EVAL_RESULTS[0] if EVAL_RESULTS else None
ANALYSIS_DIR = EVAL_PATH.parent if EVAL_PATH is not None else EXPERIMENT_DIR
PREDICTIONS_DIR = ANALYSIS_DIR / 'predictions'
EMBEDDINGS_DIR = ANALYSIS_DIR / 'embeddings'
CONFUSION_DIR = ANALYSIS_DIR / 'confusion_matrices'

config_candidates = [ANALYSIS_DIR / 'config.json', ANALYSIS_DIR / 'config.yaml', ANALYSIS_DIR / 'config.yml', EXPERIMENT_DIR / 'config.json', EXPERIMENT_DIR / 'config.yaml', EXPERIMENT_DIR / 'config.yml']
metrics_candidates = [ANALYSIS_DIR / 'metrics.json', EXPERIMENT_DIR / 'metrics.json']
summary_candidates = [ANALYSIS_DIR / 'summary.json', EXPERIMENT_DIR / 'summary.json']

config_path = next((path for path in config_candidates if path.exists()), None)
metrics_path = next((path for path in metrics_candidates if path.exists()), None)
summary_path = next((path for path in summary_candidates if path.exists()), None)

evaluation_payload = load_json(EVAL_PATH) if EVAL_PATH is not None else None
config_payload = load_mapping(config_path) if config_path is not None else None
metrics_payload = load_json(metrics_path) if metrics_path is not None else None
summary_payload = load_json(summary_path) if summary_path is not None else None

prediction_splits = discover_bundle_splits(PREDICTIONS_DIR, '_predictions.npy')
embedding_splits = discover_bundle_splits(EMBEDDINGS_DIR, '_features.npy')

prediction_bundles = {split: load_prediction_bundle(PREDICTIONS_DIR, split) for split in prediction_splits}
embedding_bundles = {split: load_embedding_bundle(EMBEDDINGS_DIR, split) for split in embedding_splits}
raw_reference = load_raw_split(DATA_ROOT, REFERENCE_SPLIT)
raw_test = load_raw_split(DATA_ROOT, TEST_SPLIT)
wavenumbers_path = DATA_ROOT / 'wavenumbers.npy'
wavenumbers = np.load(wavenumbers_path) if wavenumbers_path.exists() else None
class_name_targets = None
for candidate in [raw_reference, raw_test, next((bundle for bundle in prediction_bundles.values() if bundle is not None), None)]:
    if candidate is not None:
        class_name_targets = candidate[1] if isinstance(candidate, tuple) else candidate.get('targets')
        break
class_names = extract_class_names(config_payload or evaluation_payload, class_name_targets if class_name_targets is not None else [])

artifact_overview = pd.DataFrame([
    {'artifact': 'evaluation_json', 'path': str(EVAL_PATH) if EVAL_PATH is not None else None, 'available': EVAL_PATH is not None},
    {'artifact': 'config', 'path': str(config_path) if config_path is not None else None, 'available': config_path is not None},
    {'artifact': 'metrics', 'path': str(metrics_path) if metrics_path is not None else None, 'available': metrics_path is not None},
    {'artifact': 'summary', 'path': str(summary_path) if summary_path is not None else None, 'available': summary_path is not None},
    {'artifact': 'predictions_dir', 'path': str(PREDICTIONS_DIR), 'available': PREDICTIONS_DIR.exists()},
    {'artifact': 'embeddings_dir', 'path': str(EMBEDDINGS_DIR), 'available': EMBEDDINGS_DIR.exists()},
    {'artifact': 'raw_reference', 'path': str(DATA_ROOT / 'X_reference.npy'), 'available': raw_reference is not None},
    {'artifact': 'raw_test', 'path': str(DATA_ROOT / 'X_test.npy'), 'available': raw_test is not None},
])

display(artifact_overview)

metadata_rows = []
if isinstance(config_payload, dict):
    for key in ['stage', 'model', 'model_name', 'semantic_space', 'seed', 'notes']:
        if key in config_payload:
            metadata_rows.append({'source': 'config', 'field': key, 'value': config_payload[key]})
if isinstance(evaluation_payload, dict):
    for key in ['stage', 'model', 'model_name', 'semantic_space', 'split', 'splits']:
        if key in evaluation_payload:
            metadata_rows.append({'source': 'evaluation_json', 'field': key, 'value': evaluation_payload[key]})

if metadata_rows:
    display(pd.DataFrame(metadata_rows))
else:
    display(pd.DataFrame([{'source': 'metadata', 'field': 'status', 'value': 'No metadata fields discovered in the available files.'}]))

artifact_metric_rows = []
for name, payload in [('evaluation_json', evaluation_payload), ('metrics_json', metrics_payload), ('summary_json', summary_payload)]:
    if payload is None:
        continue
    metrics = flatten_numeric_metrics(payload)
    selected = {key: value for key, value in metrics.items() if any(token in key.lower() for token in ['acc', 'f1', 'mcc', 'roc', 'auc', 'precision', 'recall'])}
    if selected:
        row = {'artifact': name}
        row.update(selected)
        artifact_metric_rows.append(row)

if artifact_metric_rows:
    display(pd.DataFrame(artifact_metric_rows))

,artifact,path,available
0,evaluation_json,NaN,False
1,config,NaN,False
2,metrics,NaN,False
3,summary,NaN,False
4,predictions_dir,S:\raman-spectral-classifier\path_to_your_stag...,False
5,embeddings_dir,S:\raman-spectral-classifier\path_to_your_stag...,False
6,raw_reference,S:\raman-spectral-classifier\data\raw\X_refere...,True
7,raw_test,S:\raman-spectral-classifier\data\raw\X_test.npy,True


,source,field,value
0,metadata,status,No metadata fields discovered in the available...


## 2. Performance Overview

Start with the broadest quantitative question: does performance degrade on the holdout split, and is the degradation visible in both the saved evaluation artifacts and the prediction arrays? The goal here is comparison, not explanation.

In [4]:
performance_rows = []
for split, bundle in prediction_bundles.items():
    if not bundle:
        continue
    targets = bundle.get('targets')
    predictions = bundle.get('predictions')
    probabilities = bundle.get('probabilities')
    if targets is None or predictions is None:
        continue
    row = {'split': split}
    row.update(compute_split_metrics(targets, predictions, probabilities))
    performance_rows.append(row)

performance_df = pd.DataFrame(performance_rows).sort_values('split').reset_index(drop=True) if performance_rows else pd.DataFrame()
if not performance_df.empty:
    display(performance_df.round(4))

metric_candidates = [column for column in ['accuracy', 'f1_macro', 'mcc', 'precision_macro', 'recall_macro', 'roc_auc_ovr'] if column in performance_df.columns]
if metric_candidates:
    plot_df = performance_df.melt(id_vars='split', value_vars=metric_candidates, var_name='metric', value_name='value')
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
    sns.barplot(data=plot_df, x='metric', y='value', hue='split', ax=axes[0], palette='Set2')
    axes[0].set_title('Split-level metric comparison')
    axes[0].set_xlabel('')
    axes[0].set_ylabel('Score')
    axes[0].tick_params(axis='x', rotation=25)

    heatmap_df = performance_df.set_index('split')[metric_candidates]
    sns.heatmap(heatmap_df, annot=True, fmt='.3f', cmap='Blues', vmin=0, vmax=1, ax=axes[1])
    axes[1].set_title('Compact performance table')
    axes[1].set_xlabel('Metric')
    axes[1].set_ylabel('Split')
    plt.show()

json_metric_rows = []
for name, payload in [('evaluation_json', evaluation_payload), ('metrics_json', metrics_payload), ('summary_json', summary_payload)]:
    if payload is None:
        continue
    metrics = flatten_numeric_metrics(payload)
    row = {'artifact': name}
    row.update({key: value for key, value in metrics.items() if any(token in key.lower() for token in ['accuracy', 'f1', 'mcc', 'roc', 'auc'])})
    if len(row) > 1:
        json_metric_rows.append(row)

if json_metric_rows:
    display(pd.DataFrame(json_metric_rows))

## 3. Class-wise Failure Analysis

Check whether the errors are spread evenly or concentrated in a subset of classes. The focus is on ranked failures, per-class recall and F1, and the dominant incorrect destination for each source class.

In [5]:
focus_split = TEST_SPLIT if TEST_SPLIT in prediction_bundles and prediction_bundles[TEST_SPLIT] is not None else next(iter([split for split, bundle in prediction_bundles.items() if bundle is not None]), None)

if focus_split is None:
    print('No prediction bundle was found for class-wise analysis.')
else:
    focus_bundle = prediction_bundles[focus_split]
    y_true = focus_bundle['targets'].astype(int)
    y_pred = focus_bundle['predictions'].astype(int)
    class_names = extract_class_names(config_payload or evaluation_payload, y_true)
    class_failure_df, confusion_counts, label_indices, label_names = compute_class_failure_table(y_true, y_pred, class_names)

    display(class_failure_df[['class_name', 'support', 'precision', 'recall', 'f1', 'errors', 'error_rate', 'dominant_confusion_target', 'dominant_confusion_share', 'dominance_ratio']].round(4))

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
    ranked = class_failure_df.sort_values('f1', ascending=True)
    sns.barplot(data=ranked, y='class_name', x='f1', ax=axes[0], color='#3B82F6')
    axes[0].set_title(f'Sorted F1 by class ({focus_split})')
    axes[0].set_xlabel('F1')
    axes[0].set_ylabel('Class')
    axes[0].set_xlim(0, 1)

    ranked_concentration = class_failure_df.sort_values('confusion_concentration', ascending=True)
    sns.barplot(data=ranked_concentration, y='class_name', x='confusion_concentration', ax=axes[1], color='#F97316')
    axes[1].set_title('Dominant incorrect destination share')
    axes[1].set_xlabel('Share of class errors')
    axes[1].set_ylabel('Class')
    axes[1].set_xlim(0, 1)
    plt.show()

    display(class_failure_df.nsmallest(5, 'f1')[['class_name', 'support', 'precision', 'recall', 'f1', 'dominant_confusion_target', 'dominant_confusion_share']].round(4))
    display(class_failure_df.nlargest(5, 'f1')[['class_name', 'support', 'precision', 'recall', 'f1', 'dominant_confusion_target', 'dominant_confusion_share']].round(4))

No prediction bundle was found for class-wise analysis.


## 4. Confusion Matrix Forensics

Move from per-class scores to the structure of the misclassifications themselves. The question is whether a small number of directed confusion pathways dominate the matrix and whether those pathways are asymmetric.

In [6]:
if focus_split is None:
    print('No prediction bundle was found for confusion analysis.')
else:
    cm = confusion_counts.astype(float)
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_normalized = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums != 0)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7), constrained_layout=True)
    sns.heatmap(cm_normalized, cmap='mako', vmin=0, vmax=1, square=True, ax=axes[0], xticklabels=label_names, yticklabels=label_names, cbar_kws={'label': 'Row-normalized share'})
    axes[0].set_title(f'Normalized confusion matrix ({focus_split})')
    axes[0].set_xlabel('Predicted class')
    axes[0].set_ylabel('True class')
    axes[0].tick_params(axis='x', rotation=90)
    axes[0].tick_params(axis='y', rotation=0)

    if len(label_names) <= 20:
        sns.heatmap(cm, cmap='Greys', annot=False, square=True, ax=axes[1], xticklabels=label_names, yticklabels=label_names, cbar_kws={'label': 'Count'})
        axes[1].set_title('Raw confusion counts')
        axes[1].set_xlabel('Predicted class')
        axes[1].set_ylabel('True class')
        axes[1].tick_params(axis='x', rotation=90)
        axes[1].tick_params(axis='y', rotation=0)
    else:
        axes[1].axis('off')
    plt.show()

    pathway_df = confusion_pathways(confusion_counts, label_indices, class_names)
    if not pathway_df.empty:
        display(pathway_df.head(20).round(4))

        pair_rows: List[Dict[str, Any]] = []
        for source_pos, source_label in enumerate(label_indices):
            for target_pos, target_label in enumerate(label_indices):
                if target_pos <= source_pos or (source_pos == target_pos):
                    continue
                a_to_b = float(confusion_counts[source_pos, target_pos])
                b_to_a = float(confusion_counts[target_pos, source_pos])
                if a_to_b == 0 and b_to_a == 0:
                    continue
                pair_rows.append({
                    'class_a': class_names[source_label] if source_label < len(class_names) else str(source_label),
                    'class_b': class_names[target_label] if target_label < len(class_names) else str(target_label),
                    'a_to_b': a_to_b,
                    'b_to_a': b_to_a,
                    'imbalance': abs(a_to_b - b_to_a),
                    'directional_ratio': float(max(a_to_b, b_to_a) / min(a_to_b, b_to_a)) if min(a_to_b, b_to_a) > 0 else np.inf,
                })
        if pair_rows:
            pair_df = pd.DataFrame(pair_rows).sort_values(['imbalance', 'a_to_b', 'b_to_a'], ascending=[False, False, False]).reset_index(drop=True)
            display(pair_df.head(15).round(4))

No prediction bundle was found for confusion analysis.


## 5. Attractor Pathway Analysis

Identify the major source to attractor pathways. The goal is to quantify migration direction, migration rate, and the dominance of the primary incorrect destination without making causal claims too early.

In [7]:
if focus_split is None:
    print('No prediction bundle was found for attractor analysis.')
else:
    attractor_df = class_failure_df[class_failure_df['errors'] > 0].copy()
    attractor_df['migration_percentage'] = attractor_df['errors'] / attractor_df['support'].replace(0, np.nan)
    attractor_df = attractor_df.sort_values(['migration_percentage', 'dominant_confusion_share'], ascending=[False, False]).reset_index(drop=True)

    display(attractor_df[['class_name', 'support', 'errors', 'migration_percentage', 'dominant_confusion_target', 'dominant_confusion_share', 'second_confusion_share', 'dominance_ratio']].round(4))

    top_pathways = attractor_df.head(min(6, len(attractor_df)))
    if not top_pathways.empty:
        fig, ax = plt.subplots(figsize=(12, max(4, 0.55 * len(top_pathways) + 1)), constrained_layout=True)
        sns.barplot(data=top_pathways, y='class_name', x='migration_percentage', hue='dominant_confusion_target', dodge=False, ax=ax, palette='Set3')
        ax.set_title('Source to attractor migration rate')
        ax.set_xlabel('Errors as a share of source support')
        ax.set_ylabel('Source class')
        ax.set_xlim(0, max(0.05, float(top_pathways['migration_percentage'].max()) * 1.15))
        plt.show()

No prediction bundle was found for attractor analysis.


## 6. Morphology Forensics

Compare the morphology of source and attractor classes using the raw spectra from the reference and test splits. This section stays observational: mean profiles, spread, and direct difference curves.

In [8]:
if focus_split is None or raw_reference is None or raw_test is None or attractor_df.empty:
    print('Morphology analysis requires prediction bundles and both raw reference/test spectra.')
else:
    reference_X, reference_y = raw_reference
    test_X, test_y = raw_test

    def class_spectrum_stats(X: np.ndarray, y: np.ndarray, label: int) -> Tuple[np.ndarray, np.ndarray, int]:
        mask = y.astype(int) == int(label)
        spectra = X[mask]
        if spectra.size == 0:
            return np.array([]), np.array([]), 0
        return spectra.mean(axis=0), spectra.std(axis=0), int(spectra.shape[0])

    selected_pathways = attractor_df.head(min(3, len(attractor_df)))
    x_axis = wavenumbers if wavenumbers is not None and len(wavenumbers) == reference_X.shape[1] else np.arange(reference_X.shape[1])

    for _, row in selected_pathways.iterrows():
        source_label = row['class_name']
        attractor_label = row['dominant_confusion_target']
        source_index = int(row['class_index'])
        attractor_index = int(class_names.index(attractor_label)) if attractor_label in class_names else None
        if attractor_index is None:
            continue

        source_ref_mean, source_ref_std, source_ref_count = class_spectrum_stats(reference_X, reference_y, source_index)
        source_test_mean, source_test_std, source_test_count = class_spectrum_stats(test_X, test_y, source_index)
        attractor_ref_mean, attractor_ref_std, attractor_ref_count = class_spectrum_stats(reference_X, reference_y, attractor_index)

        if source_ref_count == 0 or source_test_count == 0 or attractor_ref_count == 0:
            continue

        fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
        axes[0].plot(x_axis, source_ref_mean, color='#1D4ED8', label=f'{source_label} reference')
        axes[0].fill_between(x_axis, source_ref_mean - source_ref_std, source_ref_mean + source_ref_std, color='#1D4ED8', alpha=0.15)
        axes[0].plot(x_axis, source_test_mean, color='#F97316', label=f'{source_label} test')
        axes[0].fill_between(x_axis, source_test_mean - source_test_std, source_test_mean + source_test_std, color='#F97316', alpha=0.15)
        axes[0].plot(x_axis, attractor_ref_mean, color='#059669', label=f'{attractor_label} reference')
        axes[0].fill_between(x_axis, attractor_ref_mean - attractor_ref_std, attractor_ref_mean + attractor_ref_std, color='#059669', alpha=0.15)
        axes[0].set_title(f'Mean spectra for {source_label} and attractor {attractor_label}')
        axes[0].set_xlabel('Wavenumber' if wavenumbers is not None else 'Spectral index')
        axes[0].set_ylabel('Intensity')
        axes[0].legend(loc='best')

        axes[1].plot(x_axis, source_test_mean - source_ref_mean, color='#B91C1C', label='Test minus reference for source')
        axes[1].plot(x_axis, source_ref_mean - attractor_ref_mean, color='#6D28D9', label='Source minus attractor reference')
        axes[1].axhline(0, color='black', linewidth=0.8)
        axes[1].set_title('Difference curves')
        axes[1].set_xlabel('Wavenumber' if wavenumbers is not None else 'Spectral index')
        axes[1].set_ylabel('Difference')
        axes[1].legend(loc='best')
        fig.suptitle(f'Morphology comparison for {source_label} -> {attractor_label}', y=1.02, fontsize=14, fontweight='bold')
        plt.show()

Morphology analysis requires prediction bundles and both raw reference/test spectra.


## 7. Latent Embedding Analysis

Project the Stage 1 embeddings with PCA, t-SNE, and optionally UMAP. The intent is to inspect whether overlap in representation space lines up with the confusion pathways observed above.

In [9]:
embedding_frames = []
for split, bundle in embedding_bundles.items():
    if not bundle or 'features' not in bundle or 'targets' not in bundle:
        continue
    predictions = prediction_bundles.get(split)
    pred_array = predictions['predictions'] if predictions and 'predictions' in predictions else None
    embedding_frames.append(build_embedding_frame(bundle['features'], bundle['targets'], split, class_names, pred_array))

if not embedding_frames:
    print('No embedding bundles were found for latent-space analysis.')
else:
    embedding_df = pd.concat(embedding_frames, ignore_index=True)
    feature_columns = [column for column in embedding_df.columns if column.startswith('dim_')]
    features = embedding_df[feature_columns].to_numpy()

    pca_projection = PCA(n_components=2, random_state=42).fit_transform(features)
    embedding_df['pca_1'] = pca_projection[:, 0]
    embedding_df['pca_2'] = pca_projection[:, 1]

    sample_frame = embedding_df.sample(n=min(len(embedding_df), 5000), random_state=42) if len(embedding_df) > 5000 else embedding_df.copy()
    sample_features = sample_frame[feature_columns].to_numpy()

    if len(sample_frame) > 2:
        perplexity = max(5, min(30, len(sample_frame) // 15))
        perplexity = min(perplexity, max(2, len(sample_frame) - 1))
        tsne_projection = TSNE(n_components=2, perplexity=perplexity, learning_rate='auto', init='pca', random_state=42).fit_transform(sample_features)
        sample_frame['tsne_1'] = tsne_projection[:, 0]
        sample_frame['tsne_2'] = tsne_projection[:, 1]
    else:
        sample_frame['tsne_1'] = sample_features[:, 0]
        sample_frame['tsne_2'] = np.zeros(len(sample_frame))

    if umap is not None and len(sample_frame) > 2:
        reducer = umap.UMAP(n_components=2, random_state=42)
        umap_projection = reducer.fit_transform(sample_features)
        sample_frame['umap_1'] = umap_projection[:, 0]
        sample_frame['umap_2'] = umap_projection[:, 1]
    else:
        sample_frame['umap_1'] = sample_frame['pca_1'] if 'pca_1' in sample_frame else sample_features[:, 0]
        sample_frame['umap_2'] = sample_frame['pca_2'] if 'pca_2' in sample_frame else np.zeros(len(sample_frame))

    max_classes_for_legend = 20
    if sample_frame['true_label'].nunique() > max_classes_for_legend:
        hue_class = None
    else:
        hue_class = 'true_label'

    fig, axes = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True)
    sns.scatterplot(data=sample_frame, x='pca_1', y='pca_2', hue='split', s=18, alpha=0.55, ax=axes[0, 0], palette='Set2')
    axes[0, 0].set_title('PCA colored by split')
    axes[0, 0].legend(loc='best')

    sns.scatterplot(data=sample_frame, x='pca_1', y='pca_2', hue='correct', s=18, alpha=0.55, ax=axes[0, 1], palette='Set1')
    axes[0, 1].set_title('PCA colored by correctness')
    axes[0, 1].legend(loc='best')

    sns.scatterplot(data=sample_frame, x='tsne_1', y='tsne_2', hue=hue_class, s=18, alpha=0.55, ax=axes[1, 0], palette='tab20')
    axes[1, 0].set_title('t-SNE colored by class' if hue_class else 't-SNE overview')
    axes[1, 0].legend(loc='best')

    sns.scatterplot(data=sample_frame, x='umap_1', y='umap_2', hue='correct', s=18, alpha=0.55, ax=axes[1, 1], palette='Set1')
    axes[1, 1].set_title('UMAP colored by correctness' if umap is not None else 'Projection fallback colored by correctness')
    axes[1, 1].legend(loc='best')
    plt.show()

No embedding bundles were found for latent-space analysis.


## 8. Failure Topology Analysis

Move from confusion structure to local geometry. Use centroid distances, nearest-neighbor overlap, and class-level neighborhood agreement to check whether the observed migration paths coincide with dense overlap regions in embedding space.

In [ ]:
if focus_split is None or not embedding_frames:
    print('Failure topology analysis requires both predictions and embeddings.')
else:
    test_embedding_bundle = embedding_bundles.get(TEST_SPLIT) if TEST_SPLIT in embedding_bundles else None
    ref_embedding_bundle = embedding_bundles.get(REFERENCE_SPLIT) if REFERENCE_SPLIT in embedding_bundles else None

    if test_embedding_bundle is None or ref_embedding_bundle is None:
        print('Both reference and test embedding bundles are needed for the topology section.')
    else:
        ref_features = ref_embedding_bundle['features']
        ref_targets = ref_embedding_bundle['targets'].astype(int)
        test_features = test_embedding_bundle['features']
        test_targets = test_embedding_bundle['targets'].astype(int)
        test_predictions = prediction_bundles[TEST_SPLIT]['predictions'].astype(int) if TEST_SPLIT in prediction_bundles and prediction_bundles[TEST_SPLIT] is not None else None

        labels = sorted({int(value) for value in np.concatenate([ref_targets, test_targets])})
        centroid_rows = []
        centroid_lookup: Dict[int, np.ndarray] = {}
        for label in labels:
            mask = ref_targets == label
            if mask.any():
                centroid = ref_features[mask].mean(axis=0)
                centroid_lookup[label] = centroid
                centroid_rows.append({'class_name': label_name(label, class_names), 'count': int(mask.sum())})

        if centroid_lookup:
            centroid_matrix = np.vstack([centroid_lookup[label] for label in labels if label in centroid_lookup])
            centroid_distance = pairwise_distances(centroid_matrix, centroid_matrix)
            centroid_names = [label_name(label, class_names) for label in labels if label in centroid_lookup]

            fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
            sns.heatmap(centroid_distance, cmap='viridis', xticklabels=centroid_names, yticklabels=centroid_names, ax=axes[0], cbar_kws={'label': 'Centroid distance'})
            axes[0].set_title('Reference centroid distance matrix')
            axes[0].tick_params(axis='x', rotation=90)
            axes[0].tick_params(axis='y', rotation=0)

            if test_predictions is not None:
                nn = NearestNeighbors(n_neighbors=min(15, len(ref_features)))
                nn.fit(ref_features)
                _, nn_indices = nn.kneighbors(test_features)
                neighbor_labels = ref_targets[nn_indices]
                same_label_fraction = (neighbor_labels == test_targets[:, None]).mean(axis=1)
                topology_df = pd.DataFrame({
                    'true_label': [label_name(value, class_names) for value in test_targets],
                    'pred_label': [label_name(value, class_names) for value in test_predictions],
                    'correct': test_targets == test_predictions,
                    'same_label_neighbor_fraction': same_label_fraction,
                })
                display(topology_df.groupby(['correct', 'true_label'])['same_label_neighbor_fraction'].mean().reset_index().sort_values(['correct', 'same_label_neighbor_fraction'], ascending=[False, True]).round(4))

                sns.boxplot(data=topology_df, x='correct', y='same_label_neighbor_fraction', ax=axes[1], palette='Set2')
                axes[1].set_title('Neighborhood agreement by correctness')
                axes[1].set_xlabel('Correct prediction')
                axes[1].set_ylabel('Fraction of nearest neighbors with the same true label')
                axes[1].set_ylim(0, 1)
                plt.show()
            else:
                axes[1].axis('off')
                plt.show()

            test_centroid = np.array([np.argmin([np.linalg.norm(row - centroid_lookup[label]) for label in labels if label in centroid_lookup]) for row in test_features])
            centroid_target_names = [label_name(label, class_names) for label in labels if label in centroid_lookup]
            nearest_centroid_labels = [centroid_target_names[index] for index in test_centroid]
            nearest_centroid_df = pd.DataFrame({
                'true_label': [label_name(value, class_names) for value in test_targets],
                'pred_label': [label_name(value, class_names) for value in test_predictions] if test_predictions is not None else None,
                'nearest_reference_centroid': nearest_centroid_labels,
            })
            display(nearest_centroid_df.head(25))
        else:
            print('No reference centroids could be formed from the available embeddings.')

## 9. Synthesis of Findings

Use this section to consolidate the evidence accumulated above. The notebook should summarize structured failure pathways, repeated attractor classes, morphology alignment, and latent overlap only to the extent that the observed tables and plots support those statements.

Recommended synthesis pattern:
- identify the worst-performing classes
- note whether errors are concentrated or diffuse
- list the dominant source to attractor pathways
- connect those pathways to embedding overlap and morphology comparisons
- keep causal language restrained

## 10. Notebook Boundary

This notebook stops before full representation-theory interpretation, robustness framework discussion, formal manifold analysis, domain generalization proposals, and mitigation strategies. Those belong in the later theory and robustness notebooks.

The purpose here is to organize evidence about failure behavior, not to finalize the explanation.